In [12]:
from dotenv import load_dotenv
import requests
import os
load_dotenv()
CONVEX_LOG_URL='https://kindred-mosquito-333.convex.site'
clerk_id='user_2vebGAgdrHtJFLPTxwMCXTfnfws'

def get_user_metadata(clerk_id: str) -> dict:
    url = f"{CONVEX_LOG_URL}/get-user-metadata"
    try:
        resp = requests.post(url, json={"clerk_id": clerk_id})
        if resp.status_code == 200:
            return resp.json()
        return {"status": "error", "error_message": resp.text}
    except Exception as e:
        return {"status": "error", "error_message": str(e)}
    

user_id='jn7cprzxjkx3zqjx24jd5fxndh7dyamv'

In [5]:
result = get_user_metadata(clerk_id)
import pprint
pprint.pprint(result)

{'data': {'_creationTime': 1744510864648.3171,
          '_id': 'jn7cprzxjkx3zqjx24jd5fxndh7dyamv',
          'clerkId': 'user_2vebGAgdrHtJFLPTxwMCXTfnfws',
          'dateJoined': 1744510864648,
          'demographicInfo': {'age': 23,
                              'financialStability': 5,
                              'lifeStage': 'education/training',
                              'livingArrangement': 'family',
                              'location': 'urban'},
          'email': 'anthonyfletcher@college.harvard.edu',
          'fullName': 'Anthony Fletcher',
          'interviewProgress': 0,
          'lastActive': 1744510864648},
 'success': True}


In [9]:
result['data']['demographicInfo']

{'age': 23,
 'financialStability': 5,
 'lifeStage': 'education/training',
 'livingArrangement': 'family',
 'location': 'urban'}

In [13]:
def get_progress(user_id: str) -> dict:
    """
    Queries the Convex database to retrieve the user's completed demographic fields.

    Args:
        user_id (str): The Convex user ID.

    Returns:
        dict: A dictionary mapping field names to booleans indicating completion.
    """
    url = f"{CONVEX_LOG_URL}/get-progress"
    try:
        response = requests.post(url, json={"user_id": user_id})
        if response.status_code == 200:
            return response.json()  # Expected to be the progress dictionary
        else:
            return {"status": "error", "error_message": response.text}
    except Exception as e:
        return {"status": "error", "error_message": str(e)}
item = get_progress(user_id)

In [17]:
item

{'age': True,
 'careerField': False,
 'educationLevel': False,
 'financialStability': True,
 'healthConfidence': False,
 'lifeSatisfaction': False,
 'lifeStage': True,
 'livingArrangement': True,
 'location': True,
 'socialConnection': False}

In [19]:
DEFAULT_PROGRESS = {
    'age': False,
    'lifeStage': False,
    'livingArrangement': False,
    'location': False,
    'educationLevel': False,
    'careerField': False,
    'financialStability': False,
    'socialConnection': False,
    'healthConfidence': False,
    'lifeSatisfaction': False,
}


In [20]:
import httpx
async def test_get_progress(user_id: str) -> dict:
        """
        Async query to Convex for which demographic fields are complete.
        Falls back to DEFAULT_PROGRESS on any error.
        """
        url = f"{CONVEX_LOG_URL}/get-progress"

        async with httpx.AsyncClient() as client:
            try:
                resp = await client.post(url, json={"user_id": user_id}, timeout=5.0)
                resp.raise_for_status()
                data = resp.json()
                # ensure we only return the expected keys
                return {**DEFAULT_PROGRESS, **{k: bool(data.get(k, False)) for k in DEFAULT_PROGRESS}}

            except httpx.RequestError as e:
                print(f"get_progress network error: {e}")
            except httpx.HTTPStatusError as e:
                print(f"get_progress bad status ({e.response.status_code}): {e.response.text}")
            except ValueError as e:
                print(f"get_progress invalid JSON: {e}")

        # on any failure, return the default map
        return DEFAULT_PROGRESS

await test_get_progress(user_id)


{'age': True,
 'lifeStage': True,
 'livingArrangement': True,
 'location': True,
 'educationLevel': False,
 'careerField': False,
 'financialStability': True,
 'socialConnection': False,
 'healthConfidence': False,
 'lifeSatisfaction': False}